# UIT DSC 2026 LegalIR - Step 1 Data Prep

Notebook wrapper for `legalir_step1.py`: validation, cleaning, chunking, deterministic split, official metric utilities.

This step is CPU-bound; GPU is not required.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

INPUT_ROOT = Path('/kaggle/input/datasets/ttdatto/uit-dsc26/LegalIR - Public Test')
OUTPUT_DIR = Path('/kaggle/working/step1')

SCRIPT_CANDIDATES = [
    Path('/kaggle/working/legalir_step1.py'),
    Path('/kaggle/working/step1/legalir_step1.py'),
    Path('/kaggle/input/dscuit2026-code/task1/pipeline/step1/legalir_step1.py'),
    Path('/kaggle/input/dscuit2026/task1/pipeline/step1/legalir_step1.py'),
    Path('/kaggle/input/dscuit2026-code/task1/pipeline/legalir_step1.py'),
    Path('/kaggle/input/dscuit2026/task1/pipeline/legalir_step1.py'),
    Path('legalir_step1.py'),
    Path('task1/pipeline/step1/legalir_step1.py'),
    Path('task1/pipeline/legalir_step1.py'),
]
SCRIPT_PATH = next((p for p in SCRIPT_CANDIDATES if p.exists()), None)
if SCRIPT_PATH is None:
    raise FileNotFoundError('Cannot find legalir_step1.py. Add it as a Kaggle utility script or attach this repo as a dataset.')

print('INPUT_ROOT:', INPUT_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('SCRIPT_PATH:', SCRIPT_PATH)
assert (INPUT_ROOT / 'train.json').exists()
assert (INPUT_ROOT / 'public-official.json').exists()
assert (INPUT_ROOT / 'selected-contexts').exists()

In [ ]:
cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    '--input-root', str(INPUT_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--seed', '42',
    '--dev-size', '1000',
    '--chunk-window', '320',
    '--chunk-overlap', '60',
    '--long-section-words', '900',
    '--max-chunk-warning-per-context', '420',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
report_path = OUTPUT_DIR / 'reports' / 'validation_report.json'
with report_path.open('r', encoding='utf-8') as f:
    report = json.load(f)

summary = {
    'summary': report['summary'],
    'train': report['train'],
    'public': report['public'],
    'contexts': report['contexts'],
    'split': report['split'],
    'chunks': report['chunks'],
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR), path.stat().st_size)